# 🚀 AI Dynamic Pricing — GPU Training on Google Colab

**Purpose:** Train the LSTM Demand Forecasting model and SAC RL Pricing Agent  
on a free Colab GPU (T4/A100), then download the weights and place them in your  
local project under `ai_service/models/`.

**Runtime requirement:** Go to `Runtime → Change runtime type → GPU` before running.


## 0 · Verify GPU


In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected — switch to GPU runtime!')


## 1 · Install Dependencies


In [ ]:
!pip install -q torch torchvision scikit-learn pandas numpy matplotlib seaborn scipy


## 2 · Imports & Seed


In [ ]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Ready. Device:', device)


## 3 · Dataset Generation

Generates 50,000 records (50 products × 1,000 days) with strong weekly + annual  
seasonality so the LSTM has a real temporal pattern to learn.


In [ ]:
def generate_dataset(num_samples=50000, num_products=50):
    records = []
    start = datetime(2023, 1, 1)
    spd = num_samples // num_products
    for pid in range(1, num_products + 1):
        base_p = round(random.uniform(10.0, 100.0), 2)
        cur_p  = base_p
        inv    = random.randint(500, 2000)
        base_d = random.uniform(30, 80)
        amp    = random.uniform(0.3, 0.6)
        slope  = random.uniform(-0.01, 0.02)
        for day in range(spd):
            dt  = start + timedelta(days=day)
            dow = dt.weekday()
            w   = 1.0 + amp  * np.sin(2*np.pi*dow/7.0)
            a   = 1.0 + 0.3  * np.sin(2*np.pi*day/365.0 - np.pi/2)
            t   = 1.0 + slope*(day/spd)
            promo = 1 if random.random() < 0.05 else 0
            if promo:
                cur_p = round(base_p*0.8, 2); pb = 2.5
            else:
                cur_p = round(cur_p*(1+random.uniform(-0.03,0.03)), 2)
                cur_p = max(base_p*0.7, min(cur_p, base_p*1.3))
                pb = 1.0
            pr = cur_p / base_p
            el = max(0.5, 2.0 - pr)
            tv = max(10, int(base_d*3*w*a*pb))
            td = base_d*w*a*t*el*pb
            us = max(1, int(td + np.random.normal(0, td*0.08)))
            us = min(us, inv)
            inv -= us
            if inv < 50: inv += random.randint(300,800)
            records.append({'product_id':pid,'date':dt.strftime('%Y-%m-%d'),
                'base_price':base_p,'current_price':cur_p,'traffic_views':tv,
                'units_sold':us,'day_of_week':dow,'inventory_level':inv,'promotion_flag':promo})
    df = pd.DataFrame(records)
    df = df[df['units_sold']>0].copy()
    df.sort_values(['product_id','date'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

df = generate_dataset()
print(f'Dataset shape: {df.shape}')
df.head()


## 4 · Prepare LSTM Sequences


In [ ]:
SEQ_LEN = 30   # ← GPU allows longer lookback (was 14 on CPU)

# Train on the most data-rich product
pid = df['product_id'].value_counts().index[0]
pdf = df[df['product_id']==pid].sort_values('date').reset_index(drop=True)
print(f'Training on product {pid}  |  {len(pdf)} time steps')

FEATURES = ['units_sold','current_price','traffic_views','day_of_week','promotion_flag']
scaler = MinMaxScaler()
scaled = scaler.fit_transform(pdf[FEATURES])

X, y = [], []
for i in range(len(scaled)-SEQ_LEN):
    X.append(scaled[i:i+SEQ_LEN])
    y.append(scaled[i+SEQ_LEN, 0])

X = torch.tensor(np.array(X), dtype=torch.float32)
y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(1)

split = int(len(X)*0.8)
X_train, X_test = X[:split].to(device), X[split:].to(device)
y_train, y_test = y[:split].to(device), y[split:].to(device)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


## 5 · GPU-Optimised LSTM Architecture

| Layer | Type | Units | Notes |
|-------|------|-------|-------|
| lstm1 | Bidirectional LSTM | 128×2=256 | Forward + Backward pass |
| lstm2 | Bidirectional LSTM | 64×2=128 | Stacked |
| attention | Linear | 1 | Additive self-attention |
| fc | Linear | 1 | Demand regression output |


In [ ]:
class AttentionLSTM(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, seq_len=SEQ_LEN, dropout=0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden1, batch_first=True, bidirectional=True)
        self.bn1   = nn.BatchNorm1d(seq_len)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1*2, hidden2, batch_first=True, bidirectional=True)
        self.bn2   = nn.BatchNorm1d(seq_len)
        self.drop2 = nn.Dropout(dropout)
        self.attn  = nn.Linear(hidden2*2, 1)
        self.fc    = nn.Linear(hidden2*2, 1)

    def forward(self, x):
        o1,_ = self.lstm1(x);  o1 = self.drop1(self.bn1(o1))
        o2,_ = self.lstm2(o1); o2 = self.drop2(self.bn2(o2))
        w = torch.softmax(self.attn(o2), dim=1)   # (B, T, 1)
        c = (w * o2).sum(dim=1)                   # (B, hidden2*2)
        return self.fc(c)

model = AttentionLSTM(input_dim=len(FEATURES)).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {model.__class__.__name__}  |  Trainable params: {total_params:,}')
print(model)


## 6 · Train LSTM

**GPU hyperparameters (stronger than CPU version):**

| Param | CPU value | GPU value |
|-------|-----------|-----------|
| Lookback | 14 | 30 |
| Hidden units | 64→32 | 128→64 Bidirectional |
| Epochs | 30 | 150 |
| Batch size | 32 | 256 |
| LR | 0.001 | 0.001 (AdamW) |
| Weight decay | — | 1e-4 |


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS     = 150
BATCH_SIZE = 256
LR         = 1e-3

optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion  = nn.MSELoss()
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=False)

loss_history = []
for epoch in range(1, EPOCHS+1):
    model.train()
    ep_loss = 0
    for bx, by in loader:
        optimizer.zero_grad()
        out  = model(bx)
        loss = criterion(out, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    avg = ep_loss / len(loader)
    scheduler.step()
    loss_history.append(avg)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg:.6f}')

print('Training complete!')


## 7 · Evaluate LSTM


In [ ]:
model.eval()
with torch.no_grad():
    preds_s = model(X_test).cpu().numpy()
    y_s     = y_test.cpu().numpy()

n = len(FEATURES)
def inv(arr):
    d = np.zeros((len(arr), n)); d[:,0] = arr[:,0]
    return scaler.inverse_transform(d)[:,0]

preds   = inv(preds_s)
actuals = inv(y_s)

mae  = mean_absolute_error(actuals, preds)
rmse = np.sqrt(mean_squared_error(actuals, preds))
mape = np.mean(np.abs((actuals-preds)/(actuals+1e-5)))*100
r2   = r2_score(actuals, preds)

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAPE : {mape:.2f}%')
print(f'R²   : {r2:.4f}')

# Plot
plt.figure(figsize=(12,5))
plt.plot(actuals[:150], label='Actual Demand', color='#1f77b4', lw=1.5)
plt.plot(preds[:150],   label='LSTM Forecast',  color='#ff7f0e', lw=1.5, ls='--')
plt.title('Actual vs Predicted Demand Using LSTM Model', fontsize=14)
plt.xlabel('Time (Days)', fontsize=12)
plt.ylabel('Product Demand (Units Sold)', fontsize=12)
plt.legend(); plt.grid(True, ls=':', alpha=0.6); plt.tight_layout()
plt.savefig('demand_forecast.png', dpi=300); plt.show()

# Loss curve
plt.figure(figsize=(8,4))
plt.plot(loss_history, color='#2ca02c')
plt.title('LSTM Training Loss Curve', fontsize=14)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.grid(True, ls=':', alpha=0.6); plt.tight_layout()
plt.savefig('lstm_loss_curve.png', dpi=300); plt.show()


## 8 · Save LSTM Weights


In [ ]:
torch.save(model.state_dict(), 'demand_lstm.pth')
print('✅ demand_lstm.pth saved')

# Also save the scaler so your local project can inverse-transform predictions
import pickle
with open('lstm_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✅ lstm_scaler.pkl saved')


## 9 · SAC Reinforcement Learning Agent

Soft Actor-Critic (SAC) for dynamic pricing. The agent learns a continuous pricing  
policy in the range `[base_price × 0.7, base_price × 1.3]`.


In [ ]:
# ── SAC Network Definitions ──────────────────────────────────────────────
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
        )
        self.mu_head  = nn.Linear(hidden, action_dim)
        self.log_std  = nn.Linear(hidden, action_dim)
        self.LOG_STD_MIN, self.LOG_STD_MAX = -20, 2

    def forward(self, s):
        x = self.net(s)
        mu = self.mu_head(x)
        log_std = torch.clamp(self.log_std(x), self.LOG_STD_MIN, self.LOG_STD_MAX)
        std = log_std.exp()
        dist = torch.distributions.Normal(mu, std)
        z    = dist.rsample()
        a    = torch.tanh(z)
        # log_pi with Tanh correction
        log_pi = (dist.log_prob(z) - torch.log(1 - a.pow(2) + 1e-6)).sum(dim=-1, keepdim=True)
        return a, log_pi

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        def q_net():
            return nn.Sequential(
                nn.Linear(state_dim+action_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden),               nn.ReLU(),
                nn.Linear(hidden, 1),
            )
        self.q1, self.q2 = q_net(), q_net()

    def forward(self, s, a):
        sa = torch.cat([s, a], dim=-1)
        return self.q1(sa), self.q2(sa)

import collections, random as _rnd
class ReplayBuffer:
    def __init__(self, cap=200_000): self.buf = collections.deque(maxlen=cap)
    def push(self, *args):           self.buf.append(args)
    def sample(self, n):
        batch = _rnd.sample(self.buf, n)
        return [torch.FloatTensor(np.array(x)).to(device) for x in zip(*batch)]
    def __len__(self):               return len(self.buf)


In [ ]:
# ── SAC Hyperparameters ──────────────────────────────────────────────────
STATE_DIM  = 6     # current_price, base_price, inventory, traffic, units_sold, pred_demand
ACTION_DIM = 1     # price multiplier offset in [-1, 1]
GAMMA      = 0.99
TAU        = 0.005  # soft target update
LR_ACTOR   = 3e-4
LR_CRITIC  = 3e-4
LR_ALPHA   = 3e-4
BATCH_SAC  = 256
EPISODES   = 2000   # More episodes on GPU (was 1000 on CPU)
WARMUP     = 1000   # steps before training starts

actor       = Actor(STATE_DIM, ACTION_DIM).to(device)
critic      = Critic(STATE_DIM, ACTION_DIM).to(device)
critic_tgt  = Critic(STATE_DIM, ACTION_DIM).to(device)
critic_tgt.load_state_dict(critic.state_dict())

opt_a = optim.Adam(actor.parameters(),  lr=LR_ACTOR)
opt_c = optim.Adam(critic.parameters(), lr=LR_CRITIC)

# Auto-tuned entropy coefficient α
log_alpha   = torch.zeros(1, requires_grad=True, device=device)
opt_alpha   = optim.Adam([log_alpha], lr=LR_ALPHA)
target_entr = -ACTION_DIM   # heuristic: -action_dim

buffer = ReplayBuffer()
print(f'Actor params : {sum(p.numel() for p in actor.parameters()):,}')
print(f'Critic params: {sum(p.numel() for p in critic.parameters()):,}')


In [ ]:
# ── SAC Update Function ──────────────────────────────────────────────────
def soft_update(src, tgt, tau):
    for sp, tp in zip(src.parameters(), tgt.parameters()):
        tp.data.copy_(tau*sp.data + (1-tau)*tp.data)

def sac_update():
    if len(buffer) < BATCH_SAC: return None
    s, a, r, ns, d = buffer.sample(BATCH_SAC)
    r = r.unsqueeze(1); d = d.unsqueeze(1)

    alpha = log_alpha.exp().detach()

    # Critic update
    with torch.no_grad():
        na, logp = actor(ns)
        q1t, q2t = critic_tgt(ns, na)
        q_next   = torch.min(q1t, q2t) - alpha*logp
        target_q = r + GAMMA*(1-d)*q_next
    q1, q2 = critic(s, a)
    loss_c  = nn.MSELoss()(q1, target_q) + nn.MSELoss()(q2, target_q)
    opt_c.zero_grad(); loss_c.backward(); opt_c.step()

    # Actor update
    new_a, logp = actor(s)
    q1n, q2n    = critic(s, new_a)
    loss_a      = (alpha*logp - torch.min(q1n,q2n)).mean()
    opt_a.zero_grad(); loss_a.backward(); opt_a.step()

    # Alpha update
    loss_al = -(log_alpha * (logp + target_entr).detach()).mean()
    opt_alpha.zero_grad(); loss_al.backward(); opt_alpha.step()

    soft_update(critic, critic_tgt, TAU)
    return float(loss_a.item()), float(loss_c.item())


In [ ]:
# ── SAC Training Loop ───────────────────────────────────────────────────
eval_rows   = df.tail(2000).reset_index(drop=True)
reward_hist = []

for ep in range(1, EPISODES+1):
    ep_reward = 0
    for _ in range(20):   # 20 steps per episode (more on GPU)
        idx  = _rnd.randint(14, len(eval_rows)-2)
        row  = eval_rows.iloc[idx]
        bp   = row['base_price']
        pred = row['units_sold'] * _rnd.uniform(0.9, 1.1)

        state = [row['current_price']/100, bp/100,
                 row['inventory_level']/1000, row['traffic_views']/200,
                 row['units_sold']/100,        pred/100]
        s_t   = torch.FloatTensor(state).unsqueeze(0).to(device)

        with torch.no_grad():
            a_t, _ = actor(s_t)
        mult     = torch.clamp(a_t + 1.0, 0.7, 1.3).item()   # map to [0.7, 1.3]
        new_p    = max(bp*0.7, min(row['current_price']*mult, bp*1.3))

        sales    = max(0, int(row['units_sold']*(2.0-(new_p/bp))))
        revenue  = sales * new_p
        vol_pen  = abs(mult - 1.0) * 0.5
        stk_pen  = 0.2 if sales > row['inventory_level'] else 0.0
        reward   = revenue - vol_pen - stk_pen

        next_s   = state   # simplified single-step MDP
        buffer.push(state, [mult], [reward], next_s, [0.0])
        sac_update()
        ep_reward += reward

    reward_hist.append(ep_reward)
    if ep % 200 == 0:
        avg_r = np.mean(reward_hist[-200:])
        print(f'Episode {ep:4d}/{EPISODES}  AvgReward={avg_r:.2f}  Alpha={log_alpha.exp().item():.4f}')

print('SAC Training complete!')


In [ ]:
# Plot reward curve
smooth = pd.Series(reward_hist).rolling(100).mean()
plt.figure(figsize=(10,4))
plt.plot(reward_hist, alpha=0.2, color='#9467bd', label='Raw')
plt.plot(smooth, color='#9467bd', lw=2, label='Smoothed (100-ep)')
plt.title('SAC Agent Reward Convergence Curve', fontsize=14)
plt.xlabel('Training Episode', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.legend(); plt.grid(True, ls=':', alpha=0.6); plt.tight_layout()
plt.savefig('reward_curve.png', dpi=300); plt.show()


## 10 · Save SAC Agent Weights


In [ ]:
torch.save(actor.state_dict(),      'sac_actor.pth')
torch.save(critic.state_dict(),     'sac_critic.pth')
torch.save(critic_tgt.state_dict(), 'sac_critic_target.pth')
print('✅ SAC weights saved: sac_actor.pth, sac_critic.pth, sac_critic_target.pth')


## 11 · Download Everything to Your PC

Run the cell below — Colab will prompt you to download each file.


In [ ]:
from google.colab import files
for f in ['demand_lstm.pth', 'lstm_scaler.pkl',
          'sac_actor.pth', 'sac_critic.pth', 'sac_critic_target.pth',
          'demand_forecast.png', 'lstm_loss_curve.png', 'reward_curve.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'⬇️  Downloaded: {f}')


## 12 · How to Import Weights Back into Your Local Project

After downloading, place the files as follows:

```
smart-commerce/
├── ai_service/
│   └── models/
│       ├── demand_lstm.pth        ← replace this file
│       ├── sac_actor.pth          ← new file
│       ├── sac_critic.pth         ← new file
│       └── sac_critic_target.pth  ← new file
└── paper_results/
    ├── lstm_scaler.pkl            ← place here
    └── graphs/
        ├── demand_forecast.png    ← replace
        ├── lstm_loss_curve.png    ← replace
        └── reward_curve.png       ← replace
```

### Local reload snippet (run in your terminal)

```python
# Verify LSTM loads correctly on your local machine
import torch
from paper_pipeline import AttentionLSTM
model = AttentionLSTM(input_dim=5, seq_len=30)
model.load_state_dict(torch.load('ai_service/models/demand_lstm.pth', map_location='cpu'))
model.eval()
print('LSTM loaded ✅')
```

### ⚠️  Important notes
1. The Colab model uses `seq_len=30` — update `prepare_lstm_data(sequence_length=30)` in `paper_pipeline.py`.
2. The SAC actor uses `hidden=256` — the local `SACAgent` wrapper must load `sac_actor.pth` separately.
3. Keep `lstm_scaler.pkl` — it is required to inverse-transform predictions back to original units.
